## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with SLSQP
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/"data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [7]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [8]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [9]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [10]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [11]:
num_models = X.shape[1]

In [12]:
num_models

37

In [13]:
# 2. Define the objective function with a penalty constraint
def objective_slsqp(weights):
    blend = X.values @ weights
    return 1.0 - roc_auc_score(y.values, blend)

In [14]:
# Define bounds (0 to 1 for each weight) and constraint (sum to 1)
bounds = [(0.0, 1.0) for _ in range(num_models)]
cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}

In [16]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [22]:
best_overall_score = float('inf')
best_overall_weights = None

for seed in range(10):
    np.random.seed(seed)
    raw_w = np.random.rand(num_models)
    init_weights = raw_w/np.sum(raw_w)
    
    res = minimize(
        objective_slsqp, 
        init_weights, 
        method='SLSQP', 
        bounds=bounds, 
        constraints=cons,
        options={'maxiter': 500}
    )

    if res.fun < best_overall_score:
        best_overall_score = res.fun
        best_overall_weights = res.x

In [23]:
print(f"Best Multi-Start Objective Value: {best_overall_score:.4f}")

Best Multi-Start Objective Value: 0.0527


In [24]:
# 4. Process and Normalize the Final Best Weights
best_weights = best_overall_weights

In [25]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")

Optimization Successful: True
Optimized Blending Weights: [0.015  0.0063 0.0004 0.     0.0072 0.     0.0945 0.     0.     0.0009
 0.0431 0.0055 0.0006 0.     0.0185 0.     0.     0.002  0.     0.0637
 0.     0.0026 0.     0.0576 0.0662 0.0512 0.     0.0699 0.0292 0.0925
 0.0621 0.0113 0.0553 0.1224 0.0197 0.0096 0.0929]


In [26]:
# Calculate the actual best ROC-AUC score using the optimized weights
X_blend = X.values @ best_weights
best_auc = roc_auc_score(y.values, X_blend)

print(f"Best ROC-AUC Score: {best_auc:.4f}")

Best ROC-AUC Score: 0.9473


In [27]:
# Compute the final blended predictions
final_blend = X_test.values @ best_weights

In [28]:
submit["is_fraud"] = final_blend

In [29]:
submit.to_csv(competition_path/"Blending SLSQP v4.csv",index=False)

In [30]:
res.message

'Optimization terminated successfully'